# NEC / DINN dengue baselines

Fits cumulative dengue cases (`Itot`) under the **NPC protocol**: train on the
calibration outbreak, evaluate the independent projection outbreak as hold-out.

Two PINN-style models share the same physics and loss:

- **DINN** - a single `MLP(t) -> 8 log1p states`; the 7 ODE parameters are
  trainable (`nn.Parameter`, sigmoid-bounded), `mu_h` fixed at 0.0004.
- **NEC** - adds a calibration head `baseline features -> 7 params` feeding the
  same emulation MLP; gradient flows through both nets.

Physics: full mosquito-human RHS (frequency-dependent transmission with
susceptible births); states and residuals live in log space. Loss =
`MSE(log1p Itot) + 3.0 * residual_MSE` on the calibration grid only - no
gradient from projection.

## Setup

Imports and paths. Run this once. Requires the `dinn` conda environment
(torch / scipy / pandas).

In [10]:
import warnings
warnings.filterwarnings("ignore")

In [11]:
from pathlib import Path
import sys
import importlib

import pandas as pd

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "baseline_experiments.py").exists():
    PROJECT_DIR = PROJECT_DIR / "nec_dinns"
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import baseline_experiments as be
be = importlib.reload(be)
from baseline_experiments import (
    BaselineConfig,
    default_output_dir,
    discover_regions,
    run_all_baseline_table,
    run_real_experiment,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [12]:
REGIONS = discover_regions()
MODEL_TYPES = ("dinn", "nec")
EPOCHS = 2000       # match the NPC paper training budget
SEEDS = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]   # multi-seed ensemble
# SEEDS = [0, 1]

RESULTS_DIR = default_output_dir() / "dinns_nec_results"
PREDICTIONS_DIR = default_output_dir() / "dinns_nec_predictions"
PARAMS_DIR = default_output_dir() / "dinns_nec_params"

if not REGIONS:
    raise RuntimeError(
        "No NPC data found under data. Place data_cali.csv and "
        "data_proj.csv under data/<region>/."
    )

print("regions:", REGIONS)
print("seeds  :", SEEDS)
print("results ->", RESULTS_DIR)
print("params  ->", PARAMS_DIR)


regions: ['bello', 'cam', 'iquitos', 'phi', 'sanjuan', 'vn']
seeds  : [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
results -> /home/hoang/Documents/VGU_projects/dengue_vgu/code/nec_dinns/outputs/dinns_nec_results
params  -> /home/hoang/Documents/VGU_projects/dengue_vgu/code/nec_dinns/outputs/dinns_nec_params


## Smoke test

Two epochs confirm the DINN residual path and the NEC solver-projection path.
The NEC assertion verifies projection no longer uses the time-only MLP.


In [13]:
smoke_cfg = BaselineConfig(epochs=2, log_every=1)
smoke = run_real_experiment("iquitos", "dinn", smoke_cfg, seed=0, verbose=True)
print({k: smoke[k] for k in ("seed", "loss_data_final", "loss_phys_final", "epochs_run", "train_seconds")})
print("DINN cal  MSE_norm:", round(smoke["cal_metrics"]["mse_normalized"], 4))
print("DINN proj MSE_norm:", round(smoke["proj_metrics"]["mse_normalized"], 4))
print("params :", {k: round(v, 4) for k, v in smoke["params_final"].items()})

nec_smoke = run_real_experiment("bello", "nec", smoke_cfg, seed=0, verbose=False)
print("NEC cal  MSE_norm:", round(nec_smoke["cal_metrics"]["mse_normalized"], 4))
print("NEC proj MSE_norm:", round(nec_smoke["proj_metrics"]["mse_normalized"], 4))
# NEC projects with the shared emulator MLP (no ODE solver, no IC head).
print("NEC projection method:", nec_smoke["proj_method"])

  ep     0  data=1.759e+01  phys=1.254e-01
  ep     1  data=1.751e+01  phys=1.251e-01
{'seed': 0, 'loss_data_final': 17.507871627807617, 'loss_phys_final': 0.12507696449756622, 'epochs_run': 2, 'train_seconds': 0.06702458599988859}
DINN cal  MSE_norm: 0.3513
DINN proj MSE_norm: 0.4446
params : {'lambda': 1.109, 'beta_m': 0.815, 'mu_m': 0.445, 'theta_m': 1.3617, 'beta_h': 0.7301, 'theta_h': 0.456, 'gamma_h': 0.2177, 'mu_h': 0.0004}
NEC cal  MSE_norm: 0.3298
NEC proj MSE_norm: 0.3945
NEC projection method: mlp


## Full DINN / NEC sweep

Trains every discovered region for DINN and NEC and writes one results CSV
and one predictions CSV per region (DINN and NEC rows together) under
`outputs/dinns_nec_results/<region>.csv` and
`outputs/dinns_nec_predictions/<region>.csv`. Set `RUN_FULL_SWEEP = False` to
load the last saved CSVs without retraining.

In [14]:
RUN_FULL_SWEEP = True

full_cfg = BaselineConfig(epochs=EPOCHS, log_every=500)

if RUN_FULL_SWEEP:
    results = run_all_baseline_table(
        regions=REGIONS,
        model_types=MODEL_TYPES,
        config=full_cfg,
        seeds=SEEDS,
        results_dir=RESULTS_DIR,
        predictions_dir=PREDICTIONS_DIR,
        params_dir=PARAMS_DIR,
        verbose=True,
    )
elif RESULTS_DIR.exists():
    results = pd.concat(
        (pd.read_csv(p) for p in sorted(RESULTS_DIR.glob("*.csv"))),
        ignore_index=True,
    )
else:
    results = pd.DataFrame()
    print("No saved results yet. Set RUN_FULL_SWEEP = True to train.")

display(results)


=== Bello / DINN | 10 seeds ===
  ep     0  data=2.130e+01  phys=1.221e-01
  ep   500  data=4.508e-03  phys=3.868e-03
  ep  1000  data=2.144e-03  phys=2.196e-03
  ep  1500  data=1.776e-03  phys=1.791e-03
  ep  1999  data=1.510e-03  phys=1.648e-03
  ep     0  data=2.039e+01  phys=2.670e-01
  ep   500  data=5.035e-03  phys=3.689e-03
  ep  1000  data=2.191e-03  phys=1.830e-03
  ep  1500  data=1.813e-03  phys=1.642e-03
  ep  1999  data=1.606e-03  phys=1.588e-03
  ep     0  data=2.355e+01  phys=1.879e-01
  ep   500  data=2.561e-03  phys=2.136e-03
  ep  1000  data=2.311e-03  phys=1.734e-03
  ep  1500  data=1.952e-03  phys=1.649e-03
  ep  1999  data=1.635e-03  phys=1.614e-03
  ep     0  data=2.130e+01  phys=1.550e-01
  ep   500  data=3.847e-03  phys=3.088e-03
  ep  1000  data=2.211e-03  phys=1.760e-03
  ep  1500  data=1.922e-03  phys=1.671e-03
  ep  1999  data=1.779e-03  phys=1.614e-03
  ep     0  data=2.144e+01  phys=1.033e-01
  ep   500  data=2.697e-03  phys=2.230e-03
  ep  1000  data=1.76

,region,model,n_seeds,mse_normalized,train_seconds,mse_normalized_mean,mse_normalized_std
0,bello,dinn,10,0.012880,288.187482,0.012880,0.000042
1,bello,nec,10,0.012905,345.565092,0.012905,0.000102
2,cam,dinn,10,0.004146,344.121054,0.004146,0.000156
3,cam,nec,10,0.004195,180.149539,0.004195,0.000129
4,iquitos,dinn,10,0.040254,302.993235,0.040254,0.000240
5,iquitos,nec,10,0.040207,307.953937,0.040207,0.000345
6,phi,dinn,10,0.012231,277.753279,0.012231,0.000276
7,phi,nec,10,0.012136,281.982802,0.012136,0.000152
8,sanjuan,dinn,10,0.018166,253.061248,0.018166,0.000048
9,sanjuan,nec,10,0.018178,180.712214,0.018178,0.000058
